# ANN Master Template — PyTorch (2026 Standard)

**Dataset for this walkthrough:** Fashion-MNIST (10-class image classification).
**Goal:** a reusable template for *any* ANN classification/regression problem — not
just this dataset. Swap the dataset-loading section and everything downstream keeps working.

### Why this notebook exists

Most PyTorch tutorials either (a) show you five lines of "magic" that hide what's
actually happening, or (b) show you 500 lines of production code with no explanation
of *why* any of it is there. This notebook does neither. It shows you the **full,
explicit version first** — every tensor operation visible — and only *then*
refactors into the shorter, professional version, explaining the trade-off each
time code gets shorter.

### The pipeline, at a glance

```
raw data (csv / images)
        │
        ▼
  train / val / test split   ──►  leakage-free by construction
        │
        ▼
  Dataset + DataLoader        ──►  batching, shuffling, GPU transfer
        │
        ▼
  Model (nn.Module)           ──►  Linear → BatchNorm → ReLU → Dropout, repeated
        │
        ▼
  Loss + Optimizer + Scheduler
        │
        ▼
  Training loop  ◄──────────────  validation loop (every epoch, no gradient updates)
        │
        ▼
  Early stopping + checkpointing  ──►  keep the *best* model, not the *last* one
        │
        ▼
  Optuna hyperparameter search    ──►  search over the whole pipeline above
        │
        ▼
  Retrain best config → evaluate ONCE on test set → save → inference
```

Read this top to bottom the first time. After that, treat it as a template: copy
the sections you need for a new project.


## 2. Imports

Every import below is used somewhere in this notebook — nothing is imported "just
in case." Grouped by *why* it's here, not alphabetically:

- **Data handling:** `pandas` (loading tabular/CSV data), `numpy` (array math before
  tensors)
- **Splitting & metrics:** `scikit-learn` — not because PyTorch can't split data,
  but because `train_test_split` and the metric functions are battle-tested and
  there's no reason to reimplement them
- **PyTorch core:** `torch` (tensors, autograd), `torch.nn` (layers, loss
  functions), `torch.optim` (optimizers)
- **PyTorch data pipeline:** `Dataset`, `DataLoader` — the two classes that turn
  raw arrays into GPU-ready batches
- **Mixed precision:** `torch.amp` — speeds up GPU training (Section 22)
- **Visualization:** `matplotlib` for EDA and training curves
- **Hyperparameter search:** `optuna` — installed separately below since it's not
  part of a default PyTorch environment


In [ ]:
import json
import random
import copy
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler


In [ ]:
# Optuna is not part of the default environment — install once per session.
# In Colab/Jupyter, a leading "!" runs this as a shell command.
!pip install -q optuna
import optuna
from optuna.trial import TrialState


## 3. Configuration

**Why centralize config instead of hardcoding numbers throughout the notebook?**

If `batch_size = 64` is typed into three different cells, changing it means hunting
down all three (and you *will* miss one). A single `Config` object means:

- one place to change any setting
- the exact configuration used for a run can be saved alongside the model
  (Section 26) — so a result is reproducible six months later
- Optuna (Section 23) can construct a `Config` from its suggested hyperparameters
  instead of touching global variables

We use a `dataclass` here rather than a plain dict because it gives attribute
access (`cfg.batch_size` instead of `cfg["batch_size"]`) and catches typos.


In [ ]:
@dataclass
class Config:
    # Data
    data_path: str = "fashion-mnist_train.csv"   # path to the training CSV
    test_size: float = 0.2                       # held-out fraction for val+test
    input_dim: int = 28 * 28
    num_classes: int = 10

    # Reproducibility
    seed: int = 42

    # Model architecture
    num_hidden_layers: int = 2
    neurons_per_layer: int = 128
    dropout_rate: float = 0.3

    # Training
    batch_size: int = 64
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 20
    optimizer_name: str = "Adam"          # "Adam" | "SGD" | "RMSprop"
    use_scheduler: bool = True
    use_amp: bool = True                  # mixed precision (GPU only)

    # Early stopping
    patience: int = 5

    # Misc
    checkpoint_path: str = "best_model.pt"

    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()
print(cfg)


## 4. Reproducibility

**What are we doing?** Fixing every source of randomness so re-running the
notebook produces the same result.

**Why do we need this?** Training an ANN involves several random processes:
weight initialization, data shuffling in the `DataLoader`, and dropout's random
neuron-dropping. Without a fixed seed, "did my change actually help?" becomes
unanswerable — the accuracy could differ between runs for reasons unrelated to
your change.

**What happens if we skip it?** You can't distinguish a genuine improvement from
noise. A 0.3% accuracy bump might just be a lucky shuffle.

**When should you NOT rely on this alone?** Full determinism on GPU also requires
`torch.backends.cudnn.deterministic = True`, which we set below — at a small
speed cost, since it disables some non-deterministic CUDA algorithms.


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(cfg.seed)


## 5. Device Selection — CPU vs GPU

**What are we doing?** Checking whether a CUDA-capable GPU is available and
picking it if so.

**Why?** Every tensor operation in this notebook (matrix multiplies, gradient
computation) runs faster on a GPU because GPUs run thousands of these operations
in parallel, while a CPU runs them mostly sequentially. For Fashion-MNIST with a
small ANN the difference is modest; for larger models it's the difference
between minutes and hours.

**The rule this notebook follows:** *device-agnostic code*. We never hardcode
`.cuda()` — we always use `.to(device)`, where `device` is decided once, here.
That way the exact same code runs on a laptop with no GPU or a cloud instance
with one.


In [ ]:
device = torch.device(cfg.device)
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 6. Load Dataset

**About Fashion-MNIST:** 70,000 grayscale 28×28 images of clothing items across
10 classes (t-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag,
ankle boot). It's a drop-in, slightly harder replacement for the original
MNIST digit dataset — same shape, same size, more realistic difficulty.

We load it here as a flat CSV (`label, pixel1, pixel2, ..., pixel784`), which is
the common Kaggle distribution format. **To reuse this template for a different
dataset:** replace this cell and the preprocessing in Section 8 — everything
from Section 9 onward is dataset-agnostic.


In [ ]:
df = pd.read_csv(cfg.data_path)
print(f"Shape: {df.shape}")
df.head()


## 7. Exploratory Data Analysis

**Why bother, if we already know the dataset?** In a real project you rarely do.
Two checks matter before writing a single line of model code:

1. **Do the images look right?** A reshape bug (e.g. transposed rows/columns)
   is invisible in tensor shapes but obvious the moment you plot an image.
2. **Is the class distribution balanced?** An ANN trained on 90% class-A, 10%
   class-B data will learn to just predict "A" and still score 90% accuracy —
   a trap that's invisible if you only look at accuracy after training.


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle("Sample images", fontsize=14)
for i, ax in enumerate(axes.flat):
    img = df.iloc[i, 1:].values.reshape(28, 28)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"label: {df.iloc[i, 0]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
class_counts = df.iloc[:, 0].value_counts().sort_index()
class_counts.plot(kind="bar", figsize=(8, 4), title="Class balance")
plt.xlabel("class label")
plt.ylabel("count")
plt.show()
print(class_counts)


## 8. Data Preprocessing

**What are we doing?** Splitting into features (`X`) and labels (`y`), then
scaling pixel values from `[0, 255]` to `[0, 1]`.

**Why does scaling matter?** Neural networks learn via gradient descent, and
gradient descent is sensitive to the *scale* of the inputs. A pixel value of 255
produces much larger activations (and gradients) than a pixel value of 1 would —
this can make training unstable or force you to use a tiny learning rate just to
avoid overshooting. Dividing by 255 puts every input feature on the same,
small scale, which is why normalization is close to a default step for any
neural network — not just images.

**What if we skipped it?** Training would likely still converge, but slower and
more sensitive to the learning rate you pick — one less variable to fight while
you're tuning everything else.


In [ ]:
X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values.astype(np.int64)

X = X / 255.0

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"X range after scaling: [{X.min():.2f}, {X.max():.2f}]")


## 9. Train / Validation / Test Split

**Why three sets, not two?**

- **Train**: the model's weights are updated on this data.
- **Validation**: used *during* development — to pick hyperparameters, decide
  when to stop training (Section 20), and compare architectures. The model never
  updates its weights based on validation data, but *you* (or Optuna) do make
  decisions based on it.
- **Test**: touched exactly once, at the very end (Section 25), to report a
  final, unbiased performance number.

**What is data leakage, concretely?** If you tune your dropout rate by checking
test-set accuracy, you're indirectly fitting your hyperparameters to the test
set. The reported accuracy then overestimates how the model will perform on
truly unseen data. This is why the test set is split off first and not looked at
again until Section 25 — not even for a "quick check."

**Why `stratify=y`?** It preserves the class proportions in each split, so a rare
class isn't accidentally left out of the validation set.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=cfg.test_size, random_state=cfg.seed, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=cfg.seed, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


## 10. Dataset Class — from first principles

**What is `torch.utils.data.Dataset`?** An abstraction with exactly two
responsibilities: report how many samples exist (`__len__`) and return one
sample given an index (`__getitem__`). That's it — `DataLoader` (next section)
handles batching, shuffling, and parallel loading *on top of* this simple
interface.

**Why not just pass NumPy arrays straight into the model?** Two reasons: (1) the
`Dataset`/`DataLoader` pair gives you batching and shuffling for free, and
(2) for datasets too large to fit in memory, `__getitem__` can load one file
from disk at a time instead of loading everything up front. It's not needed for
data this small, but the interface is worth using consistently, so the same
code scales to bigger datasets without changes.


In [ ]:
class TabularDataset(Dataset):
    # Wraps NumPy feature/label arrays as a PyTorch Dataset.

    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int):
        return self.features[index], self.labels[index]


train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)
test_dataset = TabularDataset(X_test, y_test)

print(f"len(train_dataset) = {len(train_dataset)}")
sample_x, sample_y = train_dataset[0]
print(f"sample feature shape: {sample_x.shape}, sample label: {sample_y.item()}")


## 11. DataLoader

**What is it doing?** Wrapping a `Dataset` to yield mini-batches instead of
single samples, optionally shuffled and loaded in parallel.

Each argument, and when to use it:

- **`batch_size`**: how many samples are averaged into a single gradient update.
  Larger batches → smoother, more stable gradients but more memory and (often)
  worse generalization; smaller batches → noisier gradients that can help escape
  sharp local minima, at the cost of slower wall-clock training.
- **`shuffle=True` (train only)**: without shuffling, the model sees samples in
  the same order every epoch, and can pick up on that order as a spurious
  pattern. Validation/test order doesn't affect the model, so we leave them
  unshuffled for reproducible evaluation.
- **`num_workers`**: spins up subprocesses to load/preprocess the next batch
  *while the GPU is still busy on the current one*. Set to 0 in a notebook by
  default for simplicity/debuggability; increase to 2–4 for real datasets with
  expensive preprocessing (e.g. loading images from disk).
- **`pin_memory=True`**: pre-allocates batches in "pinned" (non-pageable) host
  memory, which speeds up the CPU→GPU transfer. Only helps when using a GPU.
- **`persistent_workers=True`**: keeps worker subprocesses alive between epochs
  instead of respawning them — worth it when `num_workers > 0` and startup cost
  is non-trivial. No effect when `num_workers=0`.


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print(f"Batches per epoch (train): {len(train_loader)}")


## 12. Build the Model

**Why a configurable ANN instead of a fixed architecture?** So this same class
can be reused across projects and searched over by Optuna (Section 23) — the
number of layers and neurons become *inputs*, not hardcoded choices.

Each layer type, explained:

- **`nn.Linear(in, out)`**: the actual learnable transformation — a weight
  matrix and bias vector. Every "layer" below is this plus extras that help it
  train well.
- **`nn.BatchNorm1d`**: re-centers and re-scales each layer's outputs to have
  roughly zero mean and unit variance *within each mini-batch*. This keeps
  activations in a well-behaved range as they flow through many layers,
  letting you use a higher learning rate and train faster.
- **`nn.ReLU`**: the nonlinearity. Without a nonlinearity between linear layers,
  stacking them is mathematically equivalent to a *single* linear layer — no
  matter how many layers you stack, no matter how deep. Nonlinearities are
  what make "deep" mean anything.
- **`nn.Dropout(p)`**: randomly zeroes out a fraction `p` of neurons *during
  training only*. Forces the network to not rely on any single neuron too
  heavily, which reduces overfitting. Automatically disabled during
  evaluation (`model.eval()`), which is why that call matters (Section 17).

**Why `nn.Sequential` here instead of writing `forward` layer-by-layer?** The
architecture genuinely is a simple stack with no branching, so `Sequential` is
the clearest way to express that — using a manual `forward` for this specific
case would be the "unnecessary abstraction" this notebook is trying to avoid in
the *other* direction.


In [ ]:
class ANN(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        num_hidden_layers: int,
        neurons_per_layer: int,
        dropout_rate: float,
    ):
        super().__init__()

        layers = []
        current_dim = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(current_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            current_dim = neurons_per_layer

        layers.append(nn.Linear(current_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = ANN(
    input_dim=cfg.input_dim,
    num_classes=cfg.num_classes,
    num_hidden_layers=cfg.num_hidden_layers,
    neurons_per_layer=cfg.neurons_per_layer,
    dropout_rate=cfg.dropout_rate,
).to(device)

print(model)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")


## 13. Weight Initialization

**Why does the starting point of the weights matter?** If every weight starts at
the same value (or all zero), every neuron in a layer computes the *same*
gradient and stays identical forever — the network can never learn diverse
features ("symmetry"). Random initialization breaks this symmetry, but the
*scale* of that randomness still matters: too large and activations explode
through the layers; too small and they vanish, and gradients along with them.

**Kaiming (He) initialization** scales the initial weights based on the number
of inputs to a layer, specifically tuned for `ReLU` activations — which is why
we use it here, given the model uses `ReLU` throughout. (Xavier/Glorot
initialization is the equivalent choice for `tanh`/`sigmoid` activations.)

PyTorch's default `nn.Linear` initialization is already a reasonable variant of
Kaiming, so this step is often optional in practice — it's included here so the
choice is explicit and inspectable, rather than an invisible default.


In [ ]:
def kaiming_init(module: nn.Module) -> None:
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        nn.init.zeros_(module.bias)


model.apply(kaiming_init)
print("Weights re-initialized with Kaiming (He) initialization.")


## 14. Loss Function

**How to choose the right one:**

| Problem type | Loss | Final-layer output |
|---|---|---|
| Binary classification | `nn.BCEWithLogitsLoss` | 1 raw logit (sigmoid applied internally) |
| Multi-class classification (this notebook) | `nn.CrossEntropyLoss` | `num_classes` raw logits (softmax applied internally) |
| Regression | `nn.MSELoss` or `nn.L1Loss` | 1 raw value, no activation |

**Why does `CrossEntropyLoss` expect raw logits, not probabilities?** It applies
`log_softmax` internally and combines it with the negative-log-likelihood loss
in a single, numerically stable operation. If you manually apply `softmax`
*and* pass the result to `CrossEntropyLoss`, you'd be applying it twice — a
common, silent bug. This is also why the model above has **no** final
activation — the loss function expects raw scores.


In [ ]:
criterion = nn.CrossEntropyLoss()


## 15. Optimizer

**What every optimizer below is trying to do:** use the gradients from
`loss.backward()` to update each weight in the direction that reduces the loss.
They differ in *how* they use gradient history to decide the step size and
direction.

| Optimizer | Idea | When to reach for it |
|---|---|---|
| `SGD` (+ momentum) | Plain gradient descent, optionally averaging recent gradients ("momentum") to smooth out noisy updates | Well-understood, often generalizes best given enough tuning time; common in vision models when compute/time allows careful LR tuning |
| `Adam` | Keeps a per-parameter adaptive learning rate based on gradient history | Good default — fast convergence, forgiving of a less-than-ideal learning rate; the standard starting point |
| `AdamW` | Adam with weight decay implemented correctly (decoupled from the gradient update, rather than folded into it) | Prefer this over Adam whenever you're using weight decay for regularization — which is most of the time |
| `RMSprop` | Adaptive learning rate based on a moving average of *squared* gradients | Common in RNNs; less common for ANNs/CNNs now that Adam/AdamW dominate |

This template supports all three so Optuna (Section 23) can search over them,
but `AdamW` is the recommended default when you're not sure.


In [ ]:
def build_optimizer(
    model: nn.Module, name: str, lr: float, weight_decay: float
) -> optim.Optimizer:
    if name == "Adam":
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif name == "SGD":
        return optim.SGD(
            model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9
        )
    elif name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


optimizer = build_optimizer(model, cfg.optimizer_name, cfg.learning_rate, cfg.weight_decay)
print(optimizer)


## 16. Learning Rate Scheduler

**Why not just pick one learning rate and keep it fixed?** Early in training, a
larger learning rate helps you make fast progress toward a good region of the
loss landscape. Late in training, that same learning rate causes the model to
overshoot the minimum instead of settling into it. A scheduler starts high and
decreases the learning rate over time, getting the benefit of both.

We use `ReduceLROnPlateau` here specifically because it reacts to *validation
loss* rather than a fixed schedule — it only reduces the learning rate once
improvement actually stalls, which is a more robust default than guessing a
decay schedule in advance.


In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)


## 17–18. Training & Validation Loop — Stage 1: see everything

Here is the training loop with **every single operation visible**, no helper
functions yet. Read through this once, slowly — everything after this section
is a refactor of exactly this logic, never a change to it.

```
for each epoch:
    for each batch:
        optimizer.zero_grad()      # clear gradients from the previous step —
                                    # PyTorch accumulates gradients by default
        outputs = model(inputs)    # forward pass
        loss = criterion(outputs, targets)
        loss.backward()            # backward pass — computes gradients
        optimizer.step()           # apply the gradients to update weights
```

`optimizer.zero_grad()` is easy to skim past, but skipping it is a classic bug:
PyTorch *accumulates* gradients into `.grad` on every `.backward()` call by
design (useful for gradient accumulation across small batches), so without
clearing them first, each step's update would be based on the sum of every
gradient computed so far in that run — not just the current batch.

Validation runs the same forward pass with two differences: `model.eval()`
(turns off `Dropout`/`BatchNorm`'s training-time behavior) and
`torch.no_grad()` (skips building the computation graph needed for
`.backward()`, since no weights are updated here — saving memory and time).


In [ ]:
# --- Stage 1: fully explicit, single epoch shown for inspection ---

model.train()  # enables Dropout, uses batch statistics in BatchNorm
running_loss = 0.0

for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    optimizer.zero_grad()

    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * batch_features.size(0)

train_loss = running_loss / len(train_dataset)
print(f"Stage-1 demo — one training epoch, train loss: {train_loss:.4f}")

# Validation: same forward pass, no gradient updates
model.eval()  # disables Dropout, uses running statistics in BatchNorm
val_loss = 0.0
with torch.no_grad():
    for batch_features, batch_labels in val_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        val_loss += loss.item() * batch_features.size(0)

val_loss = val_loss / len(val_dataset)
print(f"Stage-1 demo — validation loss: {val_loss:.4f}")


### Stage 2: refactor into reusable functions

Now that every line above is understood, repeating this block once per epoch is
just noise. Engineers extract repeated logic into functions for one reason:
**a bug fixed in one place is fixed everywhere it's used.** If the training-step
logic were copy-pasted for every epoch, a fix would need to be applied in every
copy — and it's easy to miss one.

Nothing below is new logic — it's the exact code above, given names.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler = None,
) -> float:
    model.train()
    running_loss = 0.0

    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        if scaler is not None:
            # Mixed precision path — see Section 22 for why this differs
            with autocast(device_type=device.type):
                outputs = model(batch_features)
                loss = criterion(outputs, batch_labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * batch_features.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device
) -> float:
    model.eval()
    running_loss = 0.0

    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        running_loss += loss.item() * batch_features.size(0)

    return running_loss / len(loader.dataset)


## 19. Metrics

Loss tells you *how confidently wrong* the model is, on average — it's what
gets optimized, but it's not always what you care about reporting. These
metrics answer more concrete questions:

- **Accuracy**: what fraction of predictions are correct. Misleading on
  imbalanced classes (Section 7) — 95% accuracy is meaningless if one class
  makes up 95% of the data.
- **Precision**: of everything predicted as class X, what fraction actually was
  class X. Matters when false positives are costly.
- **Recall**: of everything that actually was class X, what fraction did the
  model catch. Matters when false negatives are costly (e.g. missing a
  defective part).
- **F1**: the harmonic mean of precision and recall — a single number when you
  need to balance both.
- **Confusion matrix**: shows *which* classes get confused for which — e.g.
  "shirt" and "coat" are visually similar in Fashion-MNIST and often confused
  with each other specifically, which plain accuracy can't tell you.

For multi-class problems, precision/recall/F1 need an averaging strategy —
we use `"macro"`, which computes the metric per class and averages unweighted,
so a rare class counts as much as a common one.


In [ ]:
@torch.no_grad()
def get_predictions(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    all_preds, all_labels = [], []

    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(device)
        outputs = model(batch_features)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch_labels.numpy())

    return np.array(all_preds), np.array(all_labels)


def compute_metrics(preds: np.ndarray, labels: np.ndarray) -> dict:
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }


## 20. Early Stopping

**What problem does this solve?** Overfitting — the point where the model keeps
improving on training data while validation performance stalls or gets worse,
because it's started memorizing training-set quirks instead of learning
generalizable patterns.

**Why `patience` instead of stopping the moment validation loss ticks up once?**
Validation loss is noisy batch-to-batch; a single worse epoch doesn't mean
overfitting has started. `patience` tolerates a few non-improving epochs before
concluding training has genuinely plateaued.


In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


## 21. Model Checkpointing

**Why save the *best* model instead of just using whatever the model looks like
after the final epoch?** Because of the overfitting curve above — the final
epoch is not necessarily the best one. If early stopping triggers at epoch 15
because validation loss stopped improving after epoch 10, epoch 10's weights are
the ones you actually want, not epoch 15's.

We save whenever validation loss reaches a new best, and reload those weights
at the end of training.


In [ ]:
def save_checkpoint(model: nn.Module, path: str) -> None:
    torch.save(model.state_dict(), path)


def load_checkpoint(model: nn.Module, path: str, device: torch.device) -> nn.Module:
    model.load_state_dict(torch.load(path, map_location=device))
    return model


## 22. Mixed Precision Training

**What is it?** Running most operations in 16-bit floating point instead of the
default 32-bit, while keeping a 32-bit master copy of the weights for stability.

**Why does this speed things up?** Modern GPUs have dedicated hardware (Tensor
Cores) that process 16-bit matrix multiplications significantly faster than
32-bit ones, and 16-bit tensors use half the memory — letting you fit larger
batches.

**Why do we need a `GradScaler`?** 16-bit floats have a much smaller
representable range than 32-bit. Small gradient values can underflow to zero in
16-bit before they'd be a problem in 32-bit. `GradScaler` multiplies the loss by
a large factor before `.backward()` (so small gradients don't underflow), then
un-scales them before the optimizer step — an implementation detail you don't
need to tune, just enable.

**When to skip this:** CPU-only training gets no benefit — mixed precision is a
GPU-specific optimization, which is why `train_one_epoch` above only activates
`autocast`/`GradScaler` when a `scaler` is passed in.


In [ ]:
use_amp = cfg.use_amp and device.type == "cuda"
scaler = GradScaler(device=device.type) if use_amp else None
print(f"Mixed precision enabled: {use_amp}")


## Putting it together: a full training run

Everything above — the epoch loop, early stopping, checkpointing, scheduler,
and mixed precision — combined into one function. This is what Section 23's
Optuna search will call repeatedly with different hyperparameters.


In [ ]:
def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    device: torch.device,
    epochs: int,
    patience: int,
    checkpoint_path: str,
    use_amp: bool = False,
    trial: "optuna.trial.Trial | None" = None,
) -> dict:
    scaler = GradScaler(device=device.type) if use_amp else None
    early_stopper = EarlyStopping(patience=patience)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss = validate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < early_stopper.best_loss:
            save_checkpoint(model, checkpoint_path)

        print(f"Epoch {epoch + 1}/{epochs} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

        # Optuna pruning: stop unpromising trials early instead of running
        # them to completion — see Section 23.
        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if early_stopper.step(val_loss):
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    load_checkpoint(model, checkpoint_path, device)
    return history


## 23. Hyperparameter Tuning with Optuna

**What is Optuna doing that manual tuning doesn't?** Instead of a grid search
(which wastes time evaluating obviously bad combinations) or random search
(which explores blindly), Optuna uses a Bayesian-style search: each trial's
result informs where it samples *next*, converging toward good regions of the
hyperparameter space faster.

**Why prune trials?** Some hyperparameter combinations are clearly underperforming
after just a few epochs — continuing to train them wastes compute. `trial.report`
+ `trial.should_prune()` (used inside `run_training` above) lets Optuna kill a
trial early if it's falling behind the best trials so far, using a `MedianPruner`.

**A bug worth calling out explicitly** (this is the mistake from the earlier
version of this notebook): every branch of the optimizer selection must
actually be *assigned* to the `optimizer` variable used in training. It's easy
to write `optim.Adam(...)` as a statement and forget the `optimizer =` in front
of it — the object gets constructed and immediately discarded, and training
silently keeps using whatever optimizer was defined earlier. The
`build_optimizer` function from Section 15 sidesteps this entirely by
`return`-ing the result instead of relying on assignment inside an `if/elif`
chain in the trial body.

**Reproducibility inside Optuna:** each trial calls `set_seed(cfg.seed)` before
building its model — otherwise two trials with *identical* suggested
hyperparameters could still get different results purely from weight-init
randomness, which would make Optuna's comparisons noisy.


In [ ]:
def objective(trial: optuna.trial.Trial) -> float:
    # Suggest hyperparameters for this trial
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 4)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 32, 256, step=32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])

    set_seed(cfg.seed)  # reproducibility across trials, see note above

    trial_train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, pin_memory=(device.type == "cuda")
    )
    trial_val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, pin_memory=(device.type == "cuda")
    )

    trial_model = ANN(
        input_dim=cfg.input_dim,
        num_classes=cfg.num_classes,
        num_hidden_layers=num_hidden_layers,
        neurons_per_layer=neurons_per_layer,
        dropout_rate=dropout_rate,
    ).to(device)
    trial_model.apply(kaiming_init)

    trial_optimizer = build_optimizer(trial_model, optimizer_name, learning_rate, weight_decay)
    trial_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        trial_optimizer, mode="min", factor=0.5, patience=2
    )

    history = run_training(
        model=trial_model,
        train_loader=trial_train_loader,
        val_loader=trial_val_loader,
        criterion=criterion,
        optimizer=trial_optimizer,
        scheduler=trial_scheduler,
        device=device,
        epochs=15,               # kept short per trial; pruning cuts weak ones sooner
        patience=4,
        checkpoint_path=f"trial_{trial.number}.pt",
        use_amp=use_amp,
        trial=trial,
    )

    return min(history["val_loss"])


study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
)
study.optimize(objective, n_trials=20)

print(f"Best val_loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


## 24. Retrain the Best Model

Optuna's trials trained on a search-time epoch budget (Section 23 used 15,
capped for search speed). Now we retrain *once* using the winning hyperparameters
with a full epoch budget and early stopping deciding when to actually stop —
this is the model that goes on to final evaluation.


In [ ]:
best_params = study.best_params
set_seed(cfg.seed)

final_train_loader = DataLoader(
    train_dataset, batch_size=best_params["batch_size"], shuffle=True,
    pin_memory=(device.type == "cuda"),
)
final_val_loader = DataLoader(
    val_dataset, batch_size=best_params["batch_size"], shuffle=False,
    pin_memory=(device.type == "cuda"),
)

final_model = ANN(
    input_dim=cfg.input_dim,
    num_classes=cfg.num_classes,
    num_hidden_layers=best_params["num_hidden_layers"],
    neurons_per_layer=best_params["neurons_per_layer"],
    dropout_rate=best_params["dropout_rate"],
).to(device)
final_model.apply(kaiming_init)

final_optimizer = build_optimizer(
    final_model, best_params["optimizer"], best_params["learning_rate"], best_params["weight_decay"]
)
final_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    final_optimizer, mode="min", factor=0.5, patience=2
)

history = run_training(
    model=final_model,
    train_loader=final_train_loader,
    val_loader=final_val_loader,
    criterion=criterion,
    optimizer=final_optimizer,
    scheduler=final_scheduler,
    device=device,
    epochs=cfg.epochs,
    patience=cfg.patience,
    checkpoint_path=cfg.checkpoint_path,
    use_amp=use_amp,
)

plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Final model training curve")
plt.show()


## 25. Final Evaluation — the test set, exactly once

This is the only cell in the entire notebook that touches `test_loader`. If you
find yourself wanting to run this cell again after changing something upstream,
that's a sign the change belongs *before* this point in the notebook (adjust
hyperparameters and go back to Section 24), not a reason to re-run this cell —
doing so is precisely the leakage described in Section 9.


In [ ]:
test_preds, test_labels = get_predictions(final_model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_labels)

print("Final test-set metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion matrix — test set")
plt.show()


## 26. Save the Model

**Why save configuration and hyperparameters alongside the weights, not just the
weights?** A `.pt` file of weights alone is useless without knowing the exact
architecture (`num_hidden_layers`, `neurons_per_layer`, ...) needed to rebuild
the model before loading those weights into it. Saving all three together means
this model can be loaded correctly a year from now without relying on memory of
"which config produced this file."


In [ ]:
artifact_dir = Path("model_artifact")
artifact_dir.mkdir(exist_ok=True)

torch.save(final_model.state_dict(), artifact_dir / "weights.pt")

with open(artifact_dir / "config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)

with open(artifact_dir / "best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)

print(f"Saved model artifact to: {artifact_dir.resolve()}")


## 27. Inference

Loading everything back from disk and predicting on a new, unseen sample — the
same steps a deployed service would run.


In [ ]:
with open(artifact_dir / "best_hyperparameters.json") as f:
    loaded_params = json.load(f)

inference_model = ANN(
    input_dim=cfg.input_dim,
    num_classes=cfg.num_classes,
    num_hidden_layers=loaded_params["num_hidden_layers"],
    neurons_per_layer=loaded_params["neurons_per_layer"],
    dropout_rate=loaded_params["dropout_rate"],
).to(device)
inference_model.load_state_dict(torch.load(artifact_dir / "weights.pt", map_location=device))
inference_model.eval()


@torch.no_grad()
def predict_single(model: nn.Module, pixels: np.ndarray, device: torch.device) -> int:
    # `pixels` is a flat array of 784 raw pixel values in [0, 255].
    x = torch.tensor(pixels / 255.0, dtype=torch.float32).unsqueeze(0).to(device)
    logits = model(x)
    return logits.argmax(dim=1).item()


# Example: predict on the first test-set sample
example_pixels = df.iloc[0, 1:].values.astype(np.float32)
predicted_class = predict_single(inference_model, example_pixels, device)
print(f"Predicted class: {predicted_class} (true label: {df.iloc[0, 0]})")


## 28. Summary

**What this template covers, end to end:**
reproducible splits → a configurable ANN → an explicit-then-refactored training
loop → early stopping and checkpointing → mixed precision → Optuna search with
pruning → a single, honest test-set evaluation → saved, reloadable artifacts.

**Limitations of this specific setup:**
- A plain ANN (stacked `Linear` layers) ignores the 2D spatial structure of
  images — a CNN would very likely outperform it on Fashion-MNIST specifically,
  since convolutions exploit the fact that nearby pixels are related. This
  notebook is a *general ANN template*, not a claim that ANNs are the right
  architecture for image data.
- `num_workers=0` in every `DataLoader` keeps the notebook simple and
  debuggable; for a dataset with expensive on-the-fly preprocessing, raising it
  would meaningfully speed up training.
- Optuna ran a fixed `n_trials=20` — for a more thorough search, either raise
  this or add a wall-clock `timeout` to `study.optimize`.

**Reusing this for a new project:** replace Sections 6 and 8 (data loading and
preprocessing) for your dataset; everything from Section 9 onward — splitting,
`Dataset`/`DataLoader`, model, training loop, Optuna search, saving, inference —
carries over unchanged.
